# Expanded-label TCN and structured event models

Private controlled experiment using the cumulative third review snapshot.
Keep the original TCN as expanded-data baseline, and test typed transitions,
additional control/fine-audio observations and 40-second context. Three fixed
ensemble seeds, 60 epochs; no validation selection. Five diagnostic training-window
folds compare seed-0 original learned-audio and short structured-rich models.
All source reviews retain draft status. No acoustic event labels are invented.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys
import torch
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/possession_structured')
WORK.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),flush=True)
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
configs=list(INPUT.rglob('structured_study_input.json'))
assert len(configs)==1
LABELS=configs[0].parent
config=json.loads(configs[0].read_text())
roots=[p.parent for p in INPUT.rglob('STRUCTURED_EXPERIMENT.md') if (p.parent/'structured_possession/experiment.py').is_file()]
assert len(roots)==1
CODE=roots[0]
base=[p for p in INPUT.rglob('features.npz') if sha(p)==config['base_features_sha256']]
audio=[p for p in INPUT.rglob('audio_cache.npz') if sha(p)==config['audio_cache_sha256']]
assert len(base)==len(audio)==1
FEATURES,AUDIO=base[0],audio[0]
RICH=LABELS/'rich_features.npz'
assert sha(RICH)==config['rich_features_sha256']
PREVIOUS={}
for arm,expected in config['previous_predictions'].items():
    paths=[p for p in INPUT.rglob('predictions.csv') if p.parent.name==arm and sha(p)==expected]
    assert len(paths)==1
    PREVIOUS[arm]=paths[0]
audit=json.loads((LABELS/'annotation_audit.json').read_text())
for name in ('train','validation'):
    assert sha(LABELS/f'{name}.csv')==audit[f'{name}_labels_sha256']
assert sum(audit['training_counts'].values())==21787
assert sum(audit['validation_counts'].values())==4078
print('Verified expanded training / unchanged validation:',audit['training_counts'],audit['validation_counts'],flush=True)


In [ ]:
DEPS=Path('/tmp/barca_structured_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
                '--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
                '--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',
         HF_HUB_OFFLINE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')


In [ ]:
REBUILT=WORK/'recreated_labels'
run('tools/import_review_draft.py','--plan',LABELS/'plan.json','--review',LABELS/'raw_review.json',
    '--train-window','train_h1_04','train_h1_05','train_h2_01','train_h2_05',
    '--validation-window','test_h1_01','--include-public-training','--allow-drafts','--out',REBUILT)
for name in ('train','validation'):
    assert sha(REBUILT/f'{name}.csv')==sha(LABELS/f'{name}.csv')
RUN=WORK/'study'
run('-m','structured_possession','compare','--features',FEATURES,'--rich-features',RICH,'--audio',AUDIO,
    '--train-labels',REBUILT/'train.csv','--validation-labels',REBUILT/'validation.csv',
    '--audit',LABELS/'annotation_audit.json','--out',RUN,'--device','cuda',
    '--epochs','60','--seeds','0','1','2','--bootstrap','1000','--cross-validate',
    '--compare',PREVIOUS['learned_audio'],PREVIOUS['tracking'],CODE/'evaluation/results/possession_v6.csv')


In [ ]:
for arm in ('factorized_short_base','factorized_short_rich','factorized_long_rich'):
    cache=FEATURES if arm.endswith('_base') else RICH
    output=RUN/arm/'regenerated.csv'
    run('-m','structured_possession','predict','--features',cache,'--audio',AUDIO,
        '--model',RUN/arm,'--out',output,'--device','cuda')
    assert output.read_bytes()==(RUN/arm/'predictions.csv').read_bytes()
    print('Byte-identical structured regeneration:',arm,sha(output),flush=True)
for arm in ('tracking','heuristic_audio','learned_audio'):
    folder=RUN/'expanded_baselines'/arm
    output=folder/'regenerated.csv'
    run('-m','temporal_possession','predict','--features',FEATURES,'--audio',AUDIO,
        '--model',folder,'--out',output,'--device','cuda')
    assert output.read_bytes()==(folder/'predictions.csv').read_bytes()
    print('Byte-identical expanded baseline regeneration:',arm,sha(output),flush=True)
report=json.loads((RUN/'report.json').read_text())
for name,score in report['scores'].items():
    print(name,'spellF1',round(score['spell_f1_mean'],4),'macroF1',round(score['macro_f1'],4),
          'D_F1',round(score['per_class']['D']['f1'],4),flush=True)
protocol=dict(config=config,all_six_global_models_regenerated=True,torch=torch.__version__,cuda=torch.version.cuda,
              gpu=torch.cuda.get_device_name(0),private_experiment=True,validation_unchanged=True,
              source_sha256={str(p.relative_to(CODE)):sha(p) for p in (CODE/'structured_possession').glob('*.py')})
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('EXPANDED STRUCTURED GPU RUN COMPLETE',flush=True)
